# Bölüm klipleri: yüz düzeltme + netleştirme (ücretsiz)

Küçük video modeli gözleri kaydırıp yüzü bozabiliyor. Bu not defteri her karede yüzü bulur,
**GFPGAN v1.4** ile gözleri ve yüz hatlarını yeniden çizer; arka planı **Real-ESRGAN x2** ile
büyütür. Sonuç 1408x832. `netlestir` not defterinin yerine geçer.

**Ayarlar:** Accelerator **GPU T4**, Internet **On**, sağ üstte **Add Input** →
`klip-uret` not defterinin çıktısı (704x416 özgün klipler). Sonra **Run All**.
Bitince Output'ta yalnızca `klipler-yuz-*.zip` olur; onu indirin.

In [ ]:
# 1) Kurulum
!pip install -q basicsr facexlib gfpgan realesrgan imageio imageio-ffmpeg
# basicsr yeni torchvision'da kaldırılan modülü arıyor; tek satırlık düzeltme
import basicsr, os
_deg = os.path.join(os.path.dirname(basicsr.__file__), "data", "degradations.py")
!sed -i 's/torchvision.transforms.functional_tensor/torchvision.transforms.functional/' {_deg}

In [ ]:
# 2) Ayarlar ve modeller
import glob, json, os, shutil, time, urllib.request, zipfile
import cv2, numpy as np, torch, imageio.v2 as imageio
from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer
from gfpgan import GFPGANer

WEIGHT = 0.6        # 0 = model yüzü olduğu gibi, 1 = tamamen yeniden çizilmiş; 0.5-0.7 titremeyi azaltır
ONLY = None         # örn. ["c03", "c19"] yalnızca bu sahneler

IN_DIR, OUT_DIR, W_DIR = "/kaggle/working/girdi", "/kaggle/working/clips", "/tmp/agirlik"
for d in (IN_DIR, OUT_DIR, W_DIR):
    os.makedirs(d, exist_ok=True)

for z in glob.glob("/kaggle/input/**/klipler-*.zip", recursive=True):
    if "-net-" in z or "-yuz-" in z:
        continue  # büyütülmüş sürümleri değil, özgün klipleri kullan
    zipfile.ZipFile(z).extractall(IN_DIR)
for f in glob.glob("/kaggle/input/**/clips/c[0-9]*.mp4", recursive=True):
    shutil.copy(f, IN_DIR)
clips = sorted(glob.glob(f"{IN_DIR}/**/c[0-9]*.mp4", recursive=True))
assert clips, "Klip bulunamadı. 'Add Input' ile klip-uret çıktısını ekleyin."
print(len(clips), "klip bulundu")

def weights(url):
    p = os.path.join(W_DIR, os.path.basename(url))  # /tmp: Output'a düşmez
    if not os.path.exists(p):
        urllib.request.urlretrieve(url, p)
    return p

bg = RealESRGANer(
    scale=2,
    model_path=weights("https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth"),
    model=RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=2),
    tile=0, half=True,
)
face = GFPGANer(
    model_path=weights("https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth"),
    upscale=2, arch="clean", channel_multiplier=2, bg_upsampler=bg,
)

In [ ]:
# 3) Her kareyi düzelt
for src in clips:
    key = os.path.splitext(os.path.basename(src))[0]
    if ONLY and key not in ONLY:
        continue
    out = os.path.join(OUT_DIR, f"{key}.mp4")
    if os.path.exists(out):
        print(f"✔ {key} zaten var"); continue
    t0 = time.time()
    r = imageio.get_reader(src)
    fps = r.get_meta_data().get("fps", 24)
    w = imageio.get_writer(out, fps=fps, codec="libx264", quality=9, macro_block_size=16)
    for fr in r:
        bgr = cv2.cvtColor(fr[:, :, :3], cv2.COLOR_RGB2BGR)
        _, _, img = face.enhance(bgr, has_aligned=False, only_center_face=False, paste_back=True, weight=WEIGHT)
        w.append_data(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    w.close(); r.close()
    torch.cuda.empty_cache()
    print(f"✔ {key}  {time.time() - t0:.0f} sn")

In [ ]:
# 4) Zip'le: Output'ta yalnızca bu dosya kalır
plans = glob.glob("/kaggle/input/**/plan.json", recursive=True)
name = "neo-genesis-2099-e1"
if plans:
    p = json.load(open(plans[0])); name = f"{p['series']}-e{p['episode']}"
zip_base = f"/kaggle/working/klipler-yuz-{name}"
shutil.make_archive(zip_base, "zip", OUT_DIR)
shutil.rmtree(IN_DIR, ignore_errors=True)
shutil.rmtree(OUT_DIR, ignore_errors=True)
for d in ("gfpgan", "weights"):
    shutil.rmtree(f"/kaggle/working/{d}", ignore_errors=True)  # facexlib'in indirdiği yüz bulma ağırlıkları
print("🎬 İndirilecek dosya:", zip_base + ".zip")